# Stage 2 — Data Processing Pipeline

This notebook combines the three data processing scripts used in Stage 2 of the car defect detection project.

| # | Script | Purpose |
|---|--------|---------|
| 1 | `process_cvat.py` | Convert CVAT annotations → YOLO format |
| 2 | `consolidate.py` | Merge multiple datasets into one unified dataset |
| 3 | `sample_and_evaluate.py` | Sample subsets & compute evaluation statistics |

---

## 1. process_cvat.py — Convert CVAT Annotations to YOLO Format

This script converts CVAT-exported COCO-format annotations into YOLO-format labels.

**Key responsibilities:**
- Reads the CVAT JSON annotation file (`instances_default.json`)
- Maps CVAT category IDs to a unified target taxonomy (dent, deform, scratch, chip, crack)
- Converts polygon segmentation coordinates to normalized YOLO bounding box format (`x_center, y_center, width, height`)
- Copies source images into the output directory
- Prints a summary of class instance counts after conversion

In [ ]:
import json
import shutil
from pathlib import Path
from collections import Counter
import numpy as np

# --- CONFIGURATION ---
JSON_PATH = Path("data/raw/cvat_golden_2000/annotations/instances_default.json")
SRC_IMAGES_DIR = Path("data/raw/cvat_golden_2000/images/default")
OUTPUT_DIR = Path("data/processed/cvat_golden_2000_yolo")

MAX_IMAGES = 2000

# --- TAXONOMY MAPPING ---
# CVAT Cat ID -> (Target Class Index, Target Class Name)
CLASS_MAPPING = {
    1: (0, "dent"),  # dent -> dent
    2: (0, "dent"),  # ding -> dent
    3: (1, "deform"),  # deform (kept separate)
    4: (2, "scratch"),  # scratch_hairline -> scratch
    5: (2, "scratch"),  # scratch_gouge -> scratch
    6: (3, "crack"),  # crack -> crack
    7: (4, "glass_shatter"),  # glass_shatter -> glass_shatter
    8: (5, "broken_lamp"),  # broken_lamp (kept separate)
    9: (6, "corrosion"),  # corrosion -> corrosion
    10: (7, "disjoint_part"),  # broken_components -> disjoint_part
}

TARGET_CLASSES = [
    "dent",  # 0
    "deform",  # 1
    "scratch",  # 2
    "crack",  # 3
    "glass_shatter",  # 4
    "broken_lamp",  # 5
    "corrosion",  # 6
    "disjoint_part",  # 7
]


def convert_dataset():
    out_images_dir = OUTPUT_DIR / "images"
    out_labels_dir = OUTPUT_DIR / "labels"
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading COCO JSON from {JSON_PATH}...")
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        coco_data = json.load(f)

    # Slice the first 2,000 images
    all_images = coco_data.get("images", [])
    selected_images = all_images[:MAX_IMAGES]
    print(
        f"Processing the first {len(selected_images)} images out of {len(all_images)} total..."
    )

    selected_img_ids = {img["id"]: img for img in selected_images}

    # Group annotations by image_id
    img_annotations = {img_id: [] for img_id in selected_img_ids.keys()}
    for ann in coco_data.get("annotations", []):
        img_id = ann.get("image_id")
        if img_id in img_annotations:
            img_annotations[img_id].append(ann)

    class_counts = Counter()
    processed_images_count = 0
    images_with_defects = 0

    for img_id, img_meta in selected_img_ids.items():
        file_name = img_meta["file_name"]
        w = float(img_meta["width"])
        h = float(img_meta["height"])

        # Check source image existence
        src_img_path = SRC_IMAGES_DIR / file_name
        if not src_img_path.exists():
            print(f"Warning: Image missing on disk: {src_img_path}")
            continue

        # Copy image to processed directory
        shutil.copy2(src_img_path, out_images_dir / file_name)

        # Process annotations for this image
        anns = img_annotations[img_id]
        txt_filename = Path(file_name).stem + ".txt"
        txt_filepath = out_labels_dir / txt_filename

        label_lines = []

        for ann in anns:
            cat_id = ann.get("category_id")
            if cat_id not in CLASS_MAPPING:
                continue

            target_cls_id, target_cls_name = CLASS_MAPPING[cat_id]
            segmentations = ann.get("segmentation", [])

            if not segmentations:
                continue

            for seg in segmentations:
                if isinstance(seg, dict) or len(seg) < 6:
                    continue

                normalized_coords = []
                for k in range(0, len(seg), 2):
                    try:
                        x_val = float(seg[k])
                        y_val = float(seg[k + 1])
                    except (ValueError, TypeError):
                        continue

                    x_norm = np.clip(x_val / w, 0.0, 1.0)
                    y_norm = np.clip(y_val / h, 0.0, 1.0)
                    normalized_coords.append(f"{x_norm:.6f} {y_norm:.6f}")

                if normalized_coords:
                    coord_str = " ".join(normalized_coords)
                    label_lines.append(f"{target_cls_id} {coord_str}")
                    class_counts[target_cls_name] += 1

        # Write label file (even if empty, to ensure 1:1 image-label mapping)
        with open(txt_filepath, "w", encoding="utf-8") as f_label:
            if label_lines:
                f_label.write("\n".join(label_lines) + "\n")
                images_with_defects += 1

        processed_images_count += 1

    # --- PRINT DETAILED SUMMARY FOR STUDYING ---
    print("\n" + "=" * 50)
    print("      DATASET CONVERSION & AUDIT SUMMARY")
    print("=" * 50)
    print(f"Total Images Processed   : {processed_images_count}")
    print(f"Images with Defects     : {images_with_defects}")
    print(f"Clean Images (0 defects): {processed_images_count - images_with_defects}")
    print("\nInstance Counts Per Class:")
    print("-" * 35)
    for cls_idx, cls_name in enumerate(TARGET_CLASSES):
        count = class_counts[cls_name]
        print(f"  [{cls_idx}] {cls_name:<18}: {count:,} instances")
    print("=" * 50)
    print(f"Output saved to: {OUTPUT_DIR.resolve()}\n")


if __name__ == "__main__":
    convert_dataset()


---

## 2. consolidate.py — Consolidate Multiple Datasets into One Unified Dataset

This script merges multiple YOLO-format datasets (from different sources) into a single consolidated dataset.

**Key responsibilities:**
- Loads a YAML config that defines source datasets, class mappings, and output paths
- Resolves class name conflicts across datasets using the configured taxonomy
- Splits the combined data into train/val/test sets with configurable ratios
- Generates a `data.yaml` file for YOLO training
- Computes and prints detailed statistics: image counts, instance counts, and mask stats per class
- Uses content-based hashing to detect and skip duplicate images

In [ ]:
import os
import json
import shutil
import random
import argparse
from pathlib import Path
from collections import defaultdict
import yaml
import hashlib


def load_config(config_name):
    """Loads the YAML configuration file with smart path resolution."""
    config_path = Path(config_name)

    script_dir = Path(__file__).resolve().parent
    config_folder_path = script_dir / "config" / config_name

    script_folder_path = script_dir / config_name

    if config_path.exists():
        target_file = config_path
    elif config_folder_path.exists():
        target_file = config_folder_path
    elif script_folder_path.exists():
        target_file = script_folder_path
    else:
        raise FileNotFoundError(
            f"Could not find '{config_name}'.\n"
            f"Checked locations:\n"
            f" 1. {config_path.resolve()}\n"
            f" 2. {config_folder_path.resolve()}\n"
            f" 3. {script_folder_path.resolve()}"
        )

    with open(target_file, "r") as file:
        print(f"[ℹ] Loaded configuration from: {target_file.resolve()}")
        return yaml.safe_load(file)


def clean_and_build_scaffolding(processed_dir):
    """Clears obsolete data and creates structured YOLO segmentation folders."""
    if processed_dir.exists():
        print(f"[⚙] Clearing obsolete data build at: {processed_dir}")
        shutil.rmtree(processed_dir)

    print("[+] Creating structured YOLO segmentation folders...")
    for split in ["train", "val", "test"]:
        (processed_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (processed_dir / split / "labels").mkdir(parents=True, exist_ok=True)


def collect_and_parse_pools(coco_json_paths, class_mapping, config):
    """Scans explicitly defined COCO JSON files and pools instances based on mapping rules and filters."""
    master_pool = []
    print("[+] Parsing COCO JSON files into a unified pool...")

    # Global tracker for max_total_instances across all images
    global_instance_counts = defaultdict(int)

    # Rarest first for stratification
    rarity_order = [
        "corrosion",
        "glass_shatter",
        "crack",
        "missing_component",
        "broken_component",
        "dent",
        "scratch",
    ]

    for json_path_str in coco_json_paths:
        json_file = Path(json_path_str)
        if not json_file.exists():
            print(f"[-] Missing JSON: {json_file}")
            continue

        # 1. Match the current JSON file to its specific dataset rules
        dataset_key = None
        for key in class_mapping.keys():
            if key in str(json_file):
                dataset_key = key
                break

        if not dataset_key:
            print(f"[-] No class mapping found for {json_file.name}. Skipping.")
            continue

        current_class_mapping = class_mapping[dataset_key]
        dataset_filters = config.get("filter_config", {}).get(dataset_key, {})
        all_rules = dataset_filters.get("__all__", {})

        split_dir = json_file.parent
        with open(json_file, "r") as f:
            coco_data = json.load(f)

        categories = {cat["id"]: cat["name"] for cat in coco_data.get("categories", [])}
        images = {img["id"]: img for img in coco_data.get("images", [])}

        img_to_anns = defaultdict(list)
        for ann in coco_data.get("annotations", []):
            img_to_anns[ann["image_id"]].append(ann)

        for img_id, anns in img_to_anns.items():
            img_info = images[img_id]
            src_img_path = split_dir / img_info["file_name"]

            if not src_img_path.exists():
                guessed_split = json_file.parent.name
                fallback_dir = json_file.parent.parent / f"{guessed_split}2017"
                src_img_path = fallback_dir / img_info["file_name"]

            if not src_img_path.exists():
                continue

            yolo_lines = []
            detected_classes_in_image = set()

            # Image-level tracker for max_instances_per_image
            image_instance_counts = defaultdict(int)

            img_w, img_h = img_info["width"], img_info["height"]
            img_area = img_w * img_h

            for ann in anns:
                raw_label = categories.get(ann["category_id"])

                # 2. Fix: Check the nested mapping for this specific dataset
                if raw_label not in current_class_mapping:
                    continue

                class_idx = current_class_mapping[raw_label]
                class_rules = dataset_filters.get(raw_label, {})

                # --- FILTER: Area Limits ---
                min_area_ratio = class_rules.get(
                    "min_area_ratio", all_rules.get("min_area_ratio", 0.0)
                )
                max_area_ratio = class_rules.get(
                    "max_area_ratio", all_rules.get("max_area_ratio", 1.0)
                )

                ann_area = ann.get("area")
                if not ann_area and "bbox" in ann:
                    ann_area = ann["bbox"][2] * ann["bbox"][3]

                area_ratio = ann_area / img_area if img_area and ann_area else 0.0

                if area_ratio < min_area_ratio or area_ratio > max_area_ratio:
                    continue  # Defect is too small or too large

                # --- FILTER: Aspect Ratio Limits ---
                max_aspect_ratio = class_rules.get(
                    "max_aspect_ratio",
                    all_rules.get("max_aspect_ratio", float("inf")),
                )
                if "bbox" in ann:
                    bw, bh = ann["bbox"][2], ann["bbox"][3]
                    if bw > 0 and bh > 0:
                        aspect_ratio = max(bw / bh, bh / bw)
                        if aspect_ratio > max_aspect_ratio:
                            continue  # Defect is too heavily skewed

                # --- FILTER: Instance Caps ---
                max_img_instances = class_rules.get(
                    "max_instances_per_image",
                    all_rules.get("max_instances_per_image", float("inf")),
                )
                max_total_instances = class_rules.get(
                    "max_total_instances",
                    all_rules.get("max_total_instances", float("inf")),
                )

                if image_instance_counts[raw_label] >= max_img_instances:
                    continue  # Image level cap reached

                global_key = f"{dataset_key}_{raw_label}"
                if global_instance_counts[global_key] >= max_total_instances:
                    continue  # Dataset global cap reached

                # 3. Validated: Increment counters and extract coordinates
                image_instance_counts[raw_label] += 1
                global_instance_counts[global_key] += 1

                segmentations = ann.get("segmentation", [])
                if not segmentations or len(segmentations) == 0:
                    if "bbox" in ann:
                        bx, by, bw, bh = ann["bbox"]
                        segmentations = [
                            [bx, by, bx + bw, by, bx + bw, by + bh, bx, by + bh]
                        ]
                    else:
                        continue

                min_points = config.get("min_polygon_points", 6)

                for seg in segmentations:
                    if len(seg) < min_points:
                        continue
                    normalized_coords = []
                    for i in range(0, len(seg), 2):
                        nx = seg[i] / img_w
                        ny = seg[i + 1] / img_h
                        if config.get("clip_coordinates", True):
                            nx = max(0.0, min(1.0, nx))
                            ny = max(0.0, min(1.0, ny))
                        normalized_coords.append(f"{nx:.6f}")
                        normalized_coords.append(f"{ny:.6f}")

                    yolo_lines.append(f"{class_idx} " + " ".join(normalized_coords))

                    standard_class = config["target_classes"][class_idx]
                    detected_classes_in_image.add(standard_class)

            if yolo_lines:
                # Group by the rarest class present in the image
                primary_class = next(
                    cls for cls in rarity_order if cls in detected_classes_in_image
                )
                master_pool.append(
                    {
                        "primary_class": primary_class,
                        "payload": {
                            "src_path": src_img_path,
                            "lines": yolo_lines,
                            "ext": os.path.splitext(img_info["file_name"])[1],
                        },
                    }
                )
    return master_pool


def parse_supervisely_dataset(supervisely_paths, class_mapping, config):
    """Scans Supervisely datasets, shuffles, and distributes 80/10/10 across pools."""
    print("[+] Parsing Supervisely JSON files with dynamic 80/10/10 split...")

    all_valid_items = []
    total_extracted_polygons = 0

    for _, dataset_root_str in supervisely_paths.items():
        dataset_root = Path(dataset_root_str)
        if not dataset_root.exists():
            print(f"[-] Missing Supervisely dataset: {dataset_root}")
            continue

        ann_files = [p for p in dataset_root.rglob("*.json") if p.parent.name == "ann"]
        print(
            f"    -> Found {len(ann_files)} annotation files in '{dataset_root.name}'"
        )

        for ann_file in ann_files:
            img_name = ann_file.name.replace(".json", "")
            img_dir = ann_file.parent.parent / "img"
            src_img_path = img_dir / img_name

            if not src_img_path.exists():
                base_name = Path(img_name).stem
                possible_images = list(img_dir.glob(f"{base_name}.*"))
                if possible_images:
                    src_img_path = possible_images[0]
                else:
                    continue

            with open(ann_file, "r") as f:
                ann_data = json.load(f)

            img_h, img_w = ann_data["size"]["height"], ann_data["size"]["width"]
            yolo_lines = []
            detected_classes_in_image = set()

            for obj in ann_data.get("objects", []):
                raw_label = obj.get("classTitle")

                if raw_label not in class_mapping:
                    continue

                class_idx = class_mapping[raw_label]
                geom_type = obj.get("geometryType")

                if geom_type != "polygon":
                    continue

                exterior = obj.get("points", {}).get("exterior", [])

                min_pairs = config.get("min_polygon_points", 6) / 2
                if len(exterior) < min_pairs:
                    continue

                normalized_coords = []
                for pt in exterior:
                    nx = pt[0] / img_w
                    ny = pt[1] / img_h
                    if config.get("clip_coordinates", True):
                        nx = max(0.0, min(1.0, nx))
                        ny = max(0.0, min(1.0, ny))
                    normalized_coords.append(f"{nx:.6f}")
                    normalized_coords.append(f"{ny:.6f}")

                yolo_lines.append(f"{class_idx} " + " ".join(normalized_coords))

                standard_class = config["target_classes"][class_idx]
                detected_classes_in_image.add(standard_class)
                total_extracted_polygons += 1

            if yolo_lines:
                rarity_order = [
                    "corrosion",
                    "glass_shatter",
                    "crack",
                    "missing_component",
                    "broken_component",
                    "dent",
                    "scratch",
                ]
                primary_class = next(
                    cls for cls in rarity_order if cls in detected_classes_in_image
                )
                all_valid_items.append(
                    {
                        "primary_class": primary_class,
                        "payload": {
                            "src_path": src_img_path,
                            "lines": yolo_lines,
                            "ext": src_img_path.suffix,
                        },
                    }
                )

    print(f"    -> Successfully extracted {total_extracted_polygons} valid polygons.")
    return all_valid_items


def deduplicate_and_split(master_pool):
    """Deduplicates images by SHA256 hash and performs a stratified 80/10/10 split."""
    print("\n[+] Deduplicating master pool via SHA256 image hashes...")
    unique_hashes = set()
    deduped_pool = []
    duplicates_removed = 0

    for item in master_pool:
        img_path = item["payload"]["src_path"]

        with open(img_path, "rb") as f:
            file_hash = hashlib.sha256(f.read()).hexdigest()

        if file_hash in unique_hashes:
            duplicates_removed += 1
            continue

        unique_hashes.add(file_hash)
        deduped_pool.append(item)

    print(f"    -> Removed {duplicates_removed} duplicate images.")
    print(f"    -> Final unique images for splitting: {len(deduped_pool)}")

    grouped_items = defaultdict(list)
    for item in deduped_pool:
        grouped_items[item["primary_class"]].append(item["payload"])

    split_pools = {
        "train": defaultdict(list),
        "val": defaultdict(list),
        "test": defaultdict(list),
    }

    print("[+] Applying 80/10/10 Stratified Split by Rarest Class...")
    for cls, items in grouped_items.items():
        random.seed(42)
        random.shuffle(items)

        total = len(items)
        train_end = int(total * 0.8)
        val_end = int(total * 0.9)

        split_pools["train"][cls].extend(items[:train_end])
        split_pools["val"][cls].extend(items[train_end:val_end])
        split_pools["test"][cls].extend(items[val_end:])

    return split_pools


def balance_and_write_dataset(split_pools, processed_dir, target_classes):
    """Applies class balancing caps and saves assets sequentially."""
    print("\n[⚙] Enforcing image-level balancing rules & archiving files...")
    global_file_counter = 0
    final_mask_stats = defaultdict(int)
    split_img_stats = defaultdict(lambda: defaultdict(int))

    for yolo_split, classes_dict in split_pools.items():
        print(f"    » Compiling split: [{yolo_split}]")
        for target_class in target_classes:
            img_list = classes_dict[target_class]

            random.seed(42)
            random.shuffle(img_list)

            split_img_stats[yolo_split][target_class] += len(img_list)

            for item in img_list:
                global_file_counter += 1
                unique_img_name = f"car_defect_{global_file_counter:06d}{item['ext']}"
                unique_lbl_name = f"car_defect_{global_file_counter:06d}.txt"

                dest_img_path = processed_dir / yolo_split / "images" / unique_img_name
                dest_lbl_path = processed_dir / yolo_split / "labels" / unique_lbl_name

                shutil.copy(item["src_path"], dest_img_path)

                with open(dest_lbl_path, "w") as lf:
                    lf.write("\n".join(item["lines"]))

                for line in item["lines"]:
                    c_idx = int(line.split()[0])
                    final_mask_stats[target_classes[c_idx]] += 1

    return final_mask_stats, split_img_stats, global_file_counter


def generate_metadata_yaml(processed_dir, target_classes):
    """Generates the official data.yaml manifest for Ultralytics tracking engines."""
    yaml_data = {
        "path": str(processed_dir.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "names": {idx: name for idx, name in enumerate(target_classes)},
    }

    output_yaml = processed_dir / "data.yaml"
    with open(output_yaml, "w") as f:
        yaml.safe_dump(yaml_data, f, default_flow_style=False)
    print(f"\n[✓] Pipeline configuration written to: {output_yaml.resolve()}")


def main():
    parser = argparse.ArgumentParser(description="YOLO Dataset Consolidation Pipeline")
    parser.add_argument(
        "--config",
        type=str,
        default="config.yaml",
        help="Path to configuration YAML file",
    )
    args = parser.parse_args()

    print("=====================================================")
    print("   INITIALIZING DATA CONSOLIDATION ROUTINE V3")
    print("=====================================================")

    # Load settings from config
    try:
        config = load_config(args.config)
    except FileNotFoundError as e:
        print(f" Error: {e}")
        return

    processed_dir = Path(config["output_processed_dir"])
    coco_json_paths = config.get("coco_json_paths", [])
    class_mapping = config["class_mapping"]

    target_classes = config["target_classes"]

    clean_and_build_scaffolding(processed_dir)

    master_pool = collect_and_parse_pools(coco_json_paths, class_mapping, config)

    supervisely_paths = config.get("supervisely_paths", {})
    if supervisely_paths:
        supervisely_pool = parse_supervisely_dataset(
            supervisely_paths, class_mapping, config
        )
        master_pool.extend(supervisely_pool)

    split_pools = deduplicate_and_split(master_pool)

    mask_stats, img_stats, total_images = balance_and_write_dataset(
        split_pools, processed_dir, target_classes
    )

    generate_metadata_yaml(processed_dir, target_classes)

    print("\n======================================================================")
    print("  📊 MLOPS DATASET DISTRIBUTION TELEMETRY")
    print("======================================================================")
    print(f"Total Unique Images Processed: {total_images:,}\n")

    print("[IMAGE COUNT PER SPLIT]")
    splits = ["train", "val", "test"]
    for i, split in enumerate(splits):
        total_in_split = sum(img_stats[split].values())
        is_last_split = i == len(splits) - 1
        split_char = "└──" if is_last_split else "├──"

        print(f"{split_char} {split.upper()} Split ───────── {total_in_split:,} images")

        for j, cls in enumerate(target_classes):
            is_last_cls = j == len(target_classes) - 1
            branch_char = "    └─" if is_last_cls else "    ├─"
            pipe_char = " " if is_last_split else "│"

            print(f"{pipe_char} {branch_char} {cls:<10}: {img_stats[split][cls]:>7,}")

    print("\n[POLYGON MASK DENSITY (ALL SPLITS)]")
    for j, cls in enumerate(target_classes):
        is_last_cls = j == len(target_classes) - 1
        branch_char = "└──" if is_last_cls else "├──"
        print(f"{branch_char} {cls:<10}: {mask_stats[cls]:>7,} instances")
    print("======================================================================\n")


if __name__ == "__main__":
    main()


---

## 3. sample_and_evaluate.py — Sample Subsets & Evaluate Dataset Statistics

This script samples a subset of data from a larger dataset and computes evaluation statistics.

**Key responsibilities:**
- Pairs images with their corresponding label files
- Calculates polygon areas from YOLO segmentation labels using the Shoelace formula
- Computes per-class statistics: number of images, total instances, and average annotation area (% of image)
- Copies sampled subsets to a new output directory
- Prints a formatted summary table of dataset composition and annotation size distribution

In [ ]:
import random
import shutil
from pathlib import Path
import yaml


def calculate_polygon_area(coords):
    n = len(coords) // 2
    area = 0.0
    for i in range(n):
        j = (i + 1) % n
        area += coords[2 * i] * coords[2 * j + 1] - coords[2 * j] * coords[2 * i + 1]
    return abs(area) / 2.0


def get_image_label_pairs(base_path):
    pairs = []
    base = Path(base_path)
    for img_path in base.rglob("*.*"):
        if img_path.suffix.lower() in [".jpg", ".png", ".jpeg"]:
            lbl_path = Path(str(img_path).replace("images", "labels")).with_suffix(
                ".txt"
            )
            if lbl_path.exists():
                pairs.append((img_path, lbl_path))
    return pairs


def stratified_sample(pairs, target_count, target_classes):
    """Samples images ensuring rare classes are prioritized."""
    # Ordered from most rare to most common
    rarity_order = [
        "corrosion",
        "crack",
        "glass_shatter",
        "missing_component",
        "broken_component",
        "scratch",
        "dent",
    ]

    grouped = {c: [] for c in rarity_order}

    # Group each image by the rarest class it contains
    for img, lbl in pairs:
        classes_in_img = set()
        with open(lbl, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    classes_in_img.add(target_classes[int(parts[0])])

        if classes_in_img:
            rarest = next(c for c in rarity_order if c in classes_in_img)
            grouped[rarest].append((img, lbl))

    selected = []
    random.seed(42)

    # Dynamically allocate the quota based on remaining images needed
    for i, cls in enumerate(rarity_order):
        random.shuffle(grouped[cls])
        available = len(grouped[cls])
        classes_left = len(rarity_order) - i
        quota = (target_count - len(selected)) // classes_left

        if available <= quota:
            selected.extend(grouped[cls])  # Take all available rare ones
        else:
            selected.extend(grouped[cls][:quota])  # Cap at quota for common ones

    return selected


def main():
    print("[⚙] Initializing Stratified 6K Sampling & Spatial Balance Check...")

    dir_prof = Path(
        "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/data/processed/stage2"
    )
    dir_custom = Path(
        "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/data/processed/stage2_custom"
    )
    dir_out = Path(
        "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/data/processed/dataset_6k"
    )

    target_classes = [
        "dent",
        "scratch",
        "crack",
        "glass_shatter",
        "broken_component",
        "missing_component",
        "corrosion",
    ]

    pairs_prof = get_image_label_pairs(dir_prof)
    pairs_custom = get_image_label_pairs(dir_custom)

    print(f"    -> Found {len(pairs_prof)} images in stage2")
    print(f"    -> Found {len(pairs_custom)} images in stage2_custom")

    # Apply Stratified Sampling
    print("[+] Applying stratification logic to prioritize rare classes...")
    sample_prof = stratified_sample(pairs_prof, 3000, target_classes)
    sample_custom = stratified_sample(pairs_custom, 3000, target_classes)
    combined = sample_prof + sample_custom

    if dir_out.exists():
        shutil.rmtree(dir_out)
    (dir_out / "images").mkdir(parents=True)
    (dir_out / "labels").mkdir(parents=True)

    stats = {c: {"instances": 0, "total_area": 0.0} for c in target_classes}
    image_counts = {c: set() for c in target_classes}

    print(
        f"[⚙] Extracting {len(combined)} total images and calculating mask footprints..."
    )

    for i, (img, lbl) in enumerate(combined):
        new_name_base = f"sample_{i:06d}"
        new_img_path = dir_out / "images" / (new_name_base + img.suffix)
        new_lbl_path = dir_out / "labels" / (new_name_base + ".txt")

        shutil.copy(img, new_img_path)
        shutil.copy(lbl, new_lbl_path)

        with open(lbl, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue

                c_idx = int(parts[0])
                coords = list(map(float, parts[1:]))

                c_name = target_classes[c_idx]
                area_fraction = calculate_polygon_area(coords)

                stats[c_name]["instances"] += 1
                stats[c_name]["total_area"] += area_fraction
                image_counts[c_name].add(new_img_path.name)

    yaml_data = {
        "path": str(dir_out.resolve()),
        "train": "images",
        "val": "images",
        "names": {idx: name for idx, name in enumerate(target_classes)},
    }
    with open(dir_out / "data.yaml", "w") as f:
        yaml.safe_dump(yaml_data, f, default_flow_style=False)

    print("\n=======================================================================")
    print("  📊 SPATIAL BALANCE & INSTANCE DISTRIBUTION (STRATIFIED 6K)")
    print("=======================================================================")
    print(
        f"{'Class':<20} | {'Images Contained':<18} | {'Total Masks':<12} | {'Avg Mask Area'}"
    )
    print("-" * 80)

    for c in target_classes:
        imgs = len(image_counts[c])
        insts = stats[c]["instances"]
        tot_a = stats[c]["total_area"]
        avg_a_pct = (tot_a / insts * 100) if insts > 0 else 0.0

        print(f"{c:<20} | {imgs:<18} | {insts:<12} | {avg_a_pct:.2f}%")
    print("=======================================================================\n")


if __name__ == "__main__":
    main()


---